# Make a small 720p copy of the tactical video

Run every cell top to bottom (**Runtime → Run all**). It asks for your Google login twice — accept both.

It downloads the tactical video from Drive into Colab's temporary disk, re-encodes it to 1280×720
**keeping every one of the 199,319 frames** (so frame *i* is still frame *i*), and saves the
result to `My Drive/mtah/BAR-ATH_tactical_720p.mp4`. Expect roughly 45–90 minutes in total.
Nothing is shared with anyone.

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

In [ ]:
from googleapiclient.discovery import build
svc = build('drive', 'v3')
IDS = ['PASTE_TACTICAL_VIDEO_ID', 'PASTE_BROADCAST_VIDEO_ID']
files = {}
for fid in IDS:
    meta = svc.files().get(fileId=fid, fields='name,size', supportsAllDrives=True).execute()
    files[fid] = meta
    print(fid, meta['name'], round(int(meta.get('size', 0)) / 1e9, 2), 'GB')
tactical = [fid for fid, m in files.items() if 'tactical' in m['name'].lower()]
assert len(tactical) == 1, 'could not tell which file is the tactical video -- check the names above'
TAC_ID = tactical[0]
print('tactical video:', files[TAC_ID]['name'])

In [ ]:
import os
from googleapiclient.http import MediaIoBaseDownload
SRC = '/content/BAR-ATH_tactical.mp4'
request = svc.files().get_media(fileId=TAC_ID, supportsAllDrives=True)
with open(SRC, 'wb') as fh:
    dl = MediaIoBaseDownload(fh, request, chunksize=200 * 1024 * 1024)
    done = False
    while not done:
        status, done = dl.next_chunk()
        print('download %3d%%' % int(status.progress() * 100), end='\r')
print('\ndownloaded', round(os.path.getsize(SRC) / 1e9, 2), 'GB')

In [ ]:
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,nb_frames,codec_name -of default=nw=1 /content/BAR-ATH_tactical.mp4

In [ ]:
# Re-encode to 720p. -vsync 0 keeps every frame exactly; no audio.
!ffmpeg -hide_banner -loglevel error -stats -y -i /content/BAR-ATH_tactical.mp4 \
    -vf scale=1280:720 -vsync 0 -an -c:v libx264 -preset veryfast -crf 28 -g 50 \
    /content/BAR-ATH_tactical_720p.mp4

In [ ]:
# Check: the frame count must be 199319.
!ffprobe -v error -count_packets -select_streams v:0 -show_entries stream=nb_read_packets,width,height -of default=nw=1 /content/BAR-ATH_tactical_720p.mp4
import os, shutil
os.makedirs('/content/drive/MyDrive/mtah', exist_ok=True)
shutil.copy('/content/BAR-ATH_tactical_720p.mp4', '/content/drive/MyDrive/mtah/BAR-ATH_tactical_720p.mp4')
print('saved to My Drive/mtah/BAR-ATH_tactical_720p.mp4,',
      round(os.path.getsize('/content/BAR-ATH_tactical_720p.mp4') / 1e9, 2), 'GB')